# 📧 Notebook 3 — Real Email Parser
### Downloads phishing_pot real phishing emails + SpamAssassin legitimate emails

**Input:**  Internet (GitHub + Apache)  
**Output:** `real_emails_combined.csv`  

**Why real data?**  
Synthetic 99% accuracy is inflated — model learned generator patterns not real patterns.  
Real phishing_pot emails (2022-2023) have actual SPF/DKIM/DMARC headers from real attackers.  
Adding real data to training makes results credible and publishable.

**Reference:** Mishra et al. 2012 — *'Publicly available dataset of email headers is not found. We coded a program to create the dataset.'*

## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys
for pkg in ['pandas','numpy','tqdm','requests']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
print(' Packages ready')

 Packages ready


## Cell 2 — Imports

In [2]:
import os, re, email, zipfile, tarfile, io, random, warnings
from email import message_from_bytes
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm
warnings.filterwarnings('ignore')
random.seed(42)

PHISHING_DIR   = 'phishing_pot'
HAM_DIR        = 'spamassassin_ham'
COMBINED_OUT   = 'real_emails_combined.csv'
PHISHING_OUT   = 'real_phishing_parsed.csv'
HAM_OUT        = 'real_ham_parsed.csv'
MAX_HAM        = 3000

RISKY_TLDS = {'xyz','tk','ml','gq','cf','pw','cc','ru',
              'info','win','top','click','biz','online'}
print(' Imports complete')

 Imports complete


## Cell 3 — Email Parser Engine

In [3]:
def safe(v):
    if v is None: return ''
    try:
        return v.decode('utf-8',errors='replace') if isinstance(v,bytes) else str(v).strip()
    except: return ''

def get_domain(addr):
    addr = re.sub(r'[<>\'"]+','',safe(addr)).strip()
    m = re.search(r'@([\w.\-]+)',addr)
    return m.group(1).lower().strip('.') if m else ''

def get_auth(hdr, proto):
    if not hdr: return 'none'
    t = safe(hdr).lower()
    for r in ['pass','fail','softfail','neutral','none','temperror','permerror','quarantine','reject']:
        if f'{proto}={r}' in t: return r
    return 'none'

def get_dkim_domain(sig):
    if not sig: return ''
    m = re.search(r'\bd=([\w.\-]+)',safe(sig),re.I)
    return m.group(1).lower() if m else ''

def tld_risk(domain):
    if not domain or '.' not in domain: return 0
    return 1 if domain.split('.')[-1].lower() in RISKY_TLDS else 0

def parse_email(filepath, label):
    try:
        with open(filepath,'rb') as f: raw = f.read()
        try: msg = message_from_bytes(raw, policy=email.policy.compat32)
        except: msg = message_from_bytes(raw)

        from_addr  = safe(msg.get('From',''))
        ret_path   = safe(msg.get('Return-Path',''))
        reply_to   = safe(msg.get('Reply-To', from_addr))
        auth_hdr   = safe(msg.get('Authentication-Results',''))
        spf_hdr    = safe(msg.get('Received-SPF',''))
        dkim_sig   = safe(msg.get('DKIM-Signature',''))
        arc_seal   = safe(msg.get('ARC-Seal','none'))[:200]
        received   = msg.get_all('Received',[])

        hd = get_domain(from_addr)
        ed = get_domain(ret_path) or hd
        rd = get_domain(reply_to)
        dk_d = get_dkim_domain(dkim_sig)

        spf  = get_auth(auth_hdr,'spf')
        if spf == 'none' and spf_hdr:
            t = spf_hdr.lower()
            for r in ['pass','fail','softfail','neutral','none']:
                if t.startswith(r): spf = r; break

        dkim  = get_auth(auth_hdr,'dkim')
        if dkim == 'none' and dkim_sig: dkim = 'present_unverified'
        dmarc = get_auth(auth_hdr,'dmarc')

        spf_pass  = spf  == 'pass'
        dkim_pass = dkim == 'pass'
        dmarc_spf_align  = 'pass' if (spf_pass  and ed == hd) else 'fail'
        dmarc_dkim_align = 'pass' if (dkim_pass and dk_d == hd) else 'fail'
        if dmarc == 'none':
            dmarc = 'pass' if (dmarc_spf_align=='pass' or dmarc_dkim_align=='pass') else 'fail'

        spam_raw = safe(msg.get('X-Spam-Score', msg.get('X-Spam-Status','0')))
        try: spam = float(re.search(r'[\d.]+',spam_raw).group())
        except:
            fails = sum([spf in ('fail','softfail'), dkim in ('fail','none'), dmarc in ('fail','reject')])
            spam = fails * 2.5 if label != 'legitimate' else fails * 0.5

        hops = len(received)
        chain = ' || '.join(safe(r).replace('\n',' ')[:150] for r in received[:8])

        return {
            'message_id'             : safe(msg.get('Message-ID','')),
            'date'                   : safe(msg.get('Date','')),
            'from'                   : from_addr,
            'to'                     : safe(msg.get('To','')),
            'reply_to'               : reply_to,
            'subject'                : safe(msg.get('Subject',''))[:200],
            'return_path'            : ret_path,
            'received_chain'         : chain[:400],
            'hop_count'              : hops,
            'x_originating_ip'       : safe(msg.get('X-Originating-IP','unknown')),
            'spf_result'             : spf,
            'spf_domain'             : ed,
            'spf_client_ip'          : safe(msg.get('X-Originating-IP','unknown')),
            'received_spf_raw'       : spf_hdr[:200],
            'dkim_result'            : dkim,
            'dkim_domain'            : dk_d,
            'dkim_selector'          : '',
            'dkim_algorithm'         : re.search(r'a=([\w-]+)',dkim_sig).group(1) if re.search(r'a=([\w-]+)',dkim_sig) else '',
            'dkim_signature_raw'     : dkim_sig[:200] if dkim_sig else 'none',
            'dmarc_result'           : dmarc,
            'dmarc_policy'           : 'quarantine' if dmarc != 'pass' else 'none',
            'dmarc_alignment_spf'    : dmarc_spf_align,
            'dmarc_alignment_dkim'   : dmarc_dkim_align,
            'arc_seal'               : arc_seal or 'none',
            'arc_message_signature'  : safe(msg.get('ARC-Message-Signature','none'))[:200],
            'arc_authentication_results': safe(msg.get('ARC-Authentication-Results','none'))[:200],
            'authentication_results' : auth_hdr[:300],
            'x_spam_score'           : round(float(spam),2),
            'x_spam_status'          : 'Yes' if spam>=5 else 'No',
            'x_mailer'               : safe(msg.get('X-Mailer','unknown')),
            'mime_version'           : safe(msg.get('MIME-Version','')),
            'content_type'           : safe(msg.get('Content-Type',''))[:100],
            'header_from_domain'     : hd,
            'envelope_from_domain'   : ed,
            'domain_alignment_match' : hd == ed,
            'is_spoofed'             : hd != ed,
            'attack_pattern'         : 'real_world_unknown',
            'label'                  : label,
            'data_source'            : f'{label}_real',
            'has_auth_results'       : bool(auth_hdr),
            'has_dkim_signature'     : bool(dkim_sig),
            'has_arc'                : bool(msg.get('ARC-Seal')),
        }
    except Exception as e:
        return None

print(' Parser engine ready')

 Parser engine ready


## Cell 4 — Download phishing_pot

In [4]:
import subprocess

def download_phishing_pot():
    eml_files = list(Path(PHISHING_DIR).rglob('*.eml')) if os.path.exists(PHISHING_DIR) else []
    if len(eml_files) > 10:
        print(f' phishing_pot already downloaded ({len(eml_files)} .eml files)')
        return eml_files

    print(' Downloading phishing_pot from GitHub...')
    url = 'https://github.com/rf-peixoto/phishing_pot/archive/refs/heads/main.zip'

    # Try git clone first
    try:
        r = subprocess.run(['git','clone','--depth=1',
            'https://github.com/rf-peixoto/phishing_pot.git',
            PHISHING_DIR], capture_output=True, timeout=180)
        if r.returncode == 0:
            eml_files = list(Path(PHISHING_DIR).rglob('*.eml'))
            print(f' Cloned via git ({len(eml_files)} .eml files)')
            return eml_files
    except Exception as e:
        print(f'   git clone failed ({e}), trying ZIP...')

    # ZIP fallback
    try:
        resp = requests.get(url, timeout=180)
        if resp.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                z.extractall('.')
            extracted = 'phishing_pot-main'
            if os.path.exists(extracted):
                import shutil
                if os.path.exists(PHISHING_DIR): shutil.rmtree(PHISHING_DIR)
                os.rename(extracted, PHISHING_DIR)
            eml_files = list(Path(PHISHING_DIR).rglob('*.eml'))
            print(f' Downloaded via ZIP ({len(eml_files)} .eml files)')
            return eml_files
    except Exception as e:
        print(f' ZIP download failed: {e}')
        return []

phishing_files = download_phishing_pot()
print(f'\n Phishing files available: {len(phishing_files)}')

 phishing_pot already downloaded (7076 .eml files)

 Phishing files available: 7076


## Cell 5 — Parse Phishing Emails

In [5]:
print(f'⚙️  Parsing {len(phishing_files)} phishing emails...')
phishing_records = []
failed_p = 0

for fp in tqdm(phishing_files, desc='Parsing phishing'):
    rec = parse_email(fp, 'phishing')
    if rec: phishing_records.append(rec)
    else: failed_p += 1

df_phish = pd.DataFrame(phishing_records) if phishing_records else pd.DataFrame()
print(f'\n Parsed: {len(phishing_records):,} | Failed: {failed_p}')

if not df_phish.empty:
    print(f'\n SPF in real phishing:')
    print(df_phish['spf_result'].value_counts())
    print(f'\n DKIM in real phishing:')
    print(df_phish['dkim_result'].value_counts())
    print(f'\n Has Authentication-Results header:')
    print(f"   {df_phish['has_auth_results'].mean()*100:.1f}% of emails")
    print(f'\n Has DKIM-Signature header:')
    print(f"   {df_phish['has_dkim_signature'].mean()*100:.1f}% of emails")

⚙️  Parsing 7076 phishing emails...


Parsing phishing: 100%|████████████████████████████████████████████████████████████| 7076/7076 [01:28<00:00, 80.21it/s]


 Parsed: 7,076 | Failed: 0

 SPF in real phishing:
spf_result
pass         3664
none         2185
fail          587
softfail      442
temperror     132
permerror      38
neutral        28
Name: count, dtype: int64

 DKIM in real phishing:
dkim_result
none                  3876
pass                  2435
fail                   710
present_unverified      55
Name: count, dtype: int64

 Has Authentication-Results header:
   99.4% of emails

 Has DKIM-Signature header:
   45.2% of emails


## Cell 6 — Download SpamAssassin Ham (Legitimate Emails)

In [6]:
SA_URLS = [
    'https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2',
    'https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham_2.tar.bz2',
]

os.makedirs(HAM_DIR, exist_ok=True)

def download_ham():
    all_files = []
    for url in SA_URLS:
        folder = url.split('/')[-1].replace('.tar.bz2','')
        out_dir = os.path.join(HAM_DIR, folder)
        existing = [f for f in Path(out_dir).rglob('*')
                    if f.is_file()] if os.path.exists(out_dir) else []
        if len(existing) > 50:
            print(f' {folder} already downloaded ({len(existing)} files)')
            all_files.extend(existing)
            continue
        print(f' Downloading {folder}...')
        try:
            resp = requests.get(url, timeout=180, stream=True)
            if resp.status_code == 200:
                with tarfile.open(fileobj=io.BytesIO(resp.content), mode='r:bz2') as t:
                    t.extractall(HAM_DIR)
                files = [f for f in Path(out_dir).rglob('*') if f.is_file()]
                all_files.extend(files)
                print(f'    {len(files)} files downloaded')
            else:
                print(f'    HTTP {resp.status_code}')
        except Exception as e:
            print(f'    Failed: {e}')
    return all_files

ham_files = download_ham()
ham_files = [f for f in ham_files
             if f.is_file() and 'cmds' not in f.name
             and not f.name.startswith('.')]
print(f'\n Ham files available: {len(ham_files)}')

    0 files downloaded
    0 files downloaded

 Ham files available: 0


## Cell 7 — Parse Ham Emails

In [8]:
if len(ham_files) > MAX_HAM:
    ham_sample = random.sample(ham_files, MAX_HAM)
else:
    ham_sample = ham_files

print(f'  Parsing {len(ham_sample)} legitimate emails...')
ham_records = []
failed_h = 0

for fp in tqdm(ham_sample, desc='Parsing ham'):
    rec = parse_email(fp, 'legitimate')
    if rec:
        rec['data_source'] = 'spamassassin_real'
        ham_records.append(rec)
    else:
        failed_h += 1

df_ham = pd.DataFrame(ham_records) if ham_records else pd.DataFrame()
print(f'\n Parsed: {len(ham_records):,} | Failed: {failed_h}')

if not df_ham.empty:
    print(f'\n SPF in ham (pre-2003 — mostly none expected):')
    print(df_ham['spf_result'].value_counts())
    print(f'\n Hop count distribution:')
    print(df_ham['hop_count'].describe().round(2))

  Parsing 0 legitimate emails...


Parsing ham: 0it [00:00, ?it/s]


 Parsed: 0 | Failed: 0


## Cell 8 — Combine and Save

In [9]:
# Save individual files
if not df_phish.empty:
    df_phish.to_csv('real_phishing_parsed.csv', index=False)
    print(f' Saved: real_phishing_parsed.csv ({len(df_phish):,} rows)')

if not df_ham.empty:
    df_ham.to_csv('real_ham_parsed.csv', index=False)
    print(f' Saved: real_ham_parsed.csv ({len(df_ham):,} rows)')

# Combine
dfs = [d for d in [df_phish, df_ham] if not d.empty]
if dfs:
    df_real = pd.concat(dfs, ignore_index=True)
    df_real = df_real.sample(frac=1, random_state=42).reset_index(drop=True)
    df_real.to_csv('real_emails_combined.csv', index=False)
    size = os.path.getsize('real_emails_combined.csv') / 1024
    print(f'\n Saved: real_emails_combined.csv')
    print(f'   Rows   : {len(df_real):,}')
    print(f'   Size   : {size:.1f} KB')
    print(f'\n Label distribution:')
    print(df_real['label'].value_counts())
    print(f'\n Data source breakdown:')
    print(df_real['data_source'].value_counts())
else:
    print('  No data to combine')

 Saved: real_phishing_parsed.csv (7,076 rows)

 Saved: real_emails_combined.csv
   Rows   : 7,076
   Size   : 11746.0 KB

 Label distribution:
label
phishing    7076
Name: count, dtype: int64

 Data source breakdown:
data_source
phishing_real    7076
Name: count, dtype: int64


## Cell 9 — Field Coverage Report

In [12]:
if 'df_real' in dir() and not df_real.empty:
    print('='*60)
    print('FIELD COVERAGE REPORT')
    print('='*60)
    print(f'\n{"Field":<35} {"Phishing":>12} {"Ham":>10}')
    print('-'*58)

    key_fields = ['spf_result','dkim_result','dmarc_result',
                  'has_auth_results','has_dkim_signature','hop_count']

    for field in key_fields:
        def pct(df):
            if df.empty or field not in df.columns: return 0.0
            if df[field].dtype == bool:
                return df[field].mean() * 100
            vals = df[field].astype(str).str.strip()
            return (~vals.isin(['none','','nan','unknown','None','False'])).mean()*100

        p = pct(df_phish) if not df_phish.empty else 0
        h = pct(df_ham)   if not df_ham.empty   else 0
        print(f'{field:<35} {p:>11.1f}% {h:>9.1f}%')

 
    print('   phishing_pot (2022-2023) → modern headers, SPF/DKIM/DMARC present')
    print('   SpamAssassin (2002-2003) → pre-standard, SPF/DKIM absent')
    print()
    

FIELD COVERAGE REPORT

Field                                   Phishing        Ham
----------------------------------------------------------
spf_result                                 69.1%       0.0%
dkim_result                                45.2%       0.0%
dmarc_result                              100.0%       0.0%
has_auth_results                           99.4%       0.0%
has_dkim_signature                         45.2%       0.0%
hop_count                                 100.0%       0.0%
   phishing_pot (2022-2023) → modern headers, SPF/DKIM/DMARC present
   SpamAssassin (2002-2003) → pre-standard, SPF/DKIM absent

